<!-- RAG with PDF data extraction  -->

In [3]:
!pip install pypdf

In [4]:
import os

from dotenv import load_dotenv
load = load_dotenv(".env")

In [5]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2202
)


In [17]:
# Ectracting PDF files
from langchain_community.document_loaders import PyPDFLoader

pdf1 = "attention.pdf"
pdf2 = "LLMForgetting.pdf"
pdf3 = "TestingAndEvaluatingLLM.pdf"
pdf4 = "user_Profile.pdf.pdf"

pdfFiles = [pdf1, pdf2, pdf3, pdf4]

documents = []

for pdf in pdfFiles:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print(f"Total number of pages in all PDFs: {len(documents)}")

Total number of pages in all PDFs: 262


In [19]:
# Text Splitting

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)

all_split_docs = text_splitter.split_documents(documents)

len(all_split_docs)  # Total number of chunks after splitting the documents


663

In [21]:
# Embedding the chunks

from langchain_core.embeddings import Embeddings
from langchain_ollama import OllamaEmbeddings

# Ollama can fail when a large document list is sent in one request.
ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")


class BatchedOllamaEmbeddings(Embeddings):
    def __init__(self, embedding_model, batch_size=8):
        self.embedding_model = embedding_model
        self.batch_size = batch_size

    def embed_documents(self, texts):
        vectors = []
        for start in range(0, len(texts), self.batch_size):
            batch = texts[start:start + self.batch_size]
            vectors.extend(self.embedding_model.embed_documents(batch))
        return vectors

    def embed_query(self, text):
        return self.embedding_model.embed_query(text)


embeddings = BatchedOllamaEmbeddings(ollama_embeddings)

vector_one = embeddings.embed_query(all_split_docs[0].page_content)
vector_two = embeddings.embed_query(all_split_docs[1].page_content)

print(len(vector_one))
print(len(vector_two))

768
768


In [44]:
# Vector store

from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=all_split_docs,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db_v3",
)

In [ ]:
# Retrieve relevant chunks

from langchain_chroma import Chroma

vector_store = Chroma(
    persist_directory="./chroma_langchain_db_v3",
    embedding_function=embeddings,
)

question = "What is my overall AI career direction?"
retrieved_docs = vector_store.similarity_search(question, k=3)

retrieved_docs

(Document(id='0fe521a6-a107-42c1-b52e-9591581873b8', metadata={'trapped': '/False', 'producer': 'ReportLab PDF Library - (opensource)', 'page': 0, 'title': 'User Knowledge Base for RAG', 'keywords': '', 'creationdate': '2026-09-07T11:56:22+00:00', 'source': 'user_Profile.pdf.pdf', 'creator': '(unspecified)', 'author': 'Generated from conversation context', 'page_label': '1', 'total_pages': 9, 'subject': '(unspecified)', 'moddate': '2026-09-07T11:56:22+00:00', 'start_index': 0}, page_content="User Knowledge Base for RAG\nPurpose: A structured knowledge document containing the non-sensitive information available from the user's profile, prior\nlearning context, projects, preferences, and goals. It is intended to be loaded into a vector database and retrieved through a\nRAG pipeline.\nImportant: This document is a synthesized knowledge base, not a verbatim transcript. It deliberately excludes sensitive\npersonal data such as health conditions, political affiliation, religion, precise loca

In [34]:
# Generate an answer from the retrieved context

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = f"""Answer the question using only the context below.
If the answer is not present in the context, say: I don't know based on the documents.

Context:
{context}

Question: {question}
Answer:"""

response = llm.invoke(prompt)
print(response.content)

The user's overall AI career direction focuses on integrating AI systems with practical software-testing workflows, emphasizing the development, testing, and evaluation of AI applications. Key areas include:  
- **AI Testing Specialization**: Prioritizing retrieval quality, hallucination detection, grounding evaluation, and security testing (e.g., prompt injection).  
- **Technical Stack**: Combining tools like **Playwright** (web automation), **WDIO + Appium** (mobile automation), **CI/CD**, **k6** (performance testing), **LangChain**, **RAG systems**, **vector databases** (e.g., ChromaDB), and **local models** (e.g., Ollama).  
- **AI-Driven Testing**: Leveraging AI for tasks like fetching QA tickets, automated reporting, and validating tool/function-calling behavior.  
- **Career Goal**: Becoming an experienced **SDET** with expertise in AI system evaluation, balancing AI development (e.g., LLM applications) and rigorous testing frameworks.  

This direction merges software testing 

In [48]:
# Create a profile-specific retriever

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": {"source": "user_Profile.pdf.pdf"},
    },
)

retriever.invoke("What programming languages are listed in my profile?")

[Document(id='124b245b-2de1-4f46-a7b9-15737d1cb3b3', metadata={'start_index': 815, 'page_label': '1', 'subject': '(unspecified)', 'author': 'Generated from conversation context', 'trapped': '/False', 'keywords': '', 'moddate': '2026-09-07T11:56:22+00:00', 'producer': 'ReportLab PDF Library - (opensource)', 'creationdate': '2026-09-07T11:56:22+00:00', 'page': 0, 'creator': '(unspecified)', 'source': 'user_Profile.pdf.pdf', 'title': 'User Knowledge Base for RAG', 'total_pages': 9}, page_content='Programming languages: Java, JavaScript, TypeScript.\nCore automation background: Appium with Java and Page Object Model (POM). The user is also working with\nWebdriverIO, Appium v2, TypeScript, Playwright, CI/CD, reporting, and test architecture.\n2. Current Technical Direction\nThe user is moving toward a combined profile centered on advanced test automation plus AI\ntesting/evaluation and AI application development.\n\x7f\nAI testing/evaluation: approximately 70% of the intended AI focus.\n\x7

In [49]:
# Full RetrievalQA implementation

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

question = "What programming languages are listed in my profile?"

prompt = ChatPromptTemplate.from_template("""Use only the context below to answer the question.
If the answer is not in the context, say: I don't know based on the documents.
Keep the answer concise and do not add unsupported details.

Context:
{context}

Question: {question}
Answer:""")


def format_documents(documents):
    return "\n\n".join(document.page_content for document in documents)


retrieval_qa_chain = (
    {
        "context": retriever | format_documents,
        "question": lambda value: value,
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = retrieval_qa_chain.invoke(question)
print(answer)

Java, JavaScript, TypeScript.


In [ ]:
# RetrivalQA

